<div align='center'>

# 🔭 Vision Transformer for Image Classification
### Running on Google Colab · CIFAR-10 · PyTorch · GPU Accelerated

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

---
</div>

### 📋 What this notebook does:
1. **Sets up** the complete ViT project structure on Colab
2. **Installs** all required packages
3. **Trains** Vision Transformer on CIFAR-10 (with GPU)
4. **Evaluates** on the test set & shows metrics
5. **Visualises** training curves, confusion matrix, predictions
6. **Downloads** trained model & plots to your Google Drive

> ⚡ **Enable GPU first!**  
> `Runtime` → `Change runtime type` → `Hardware Accelerator` → `GPU (T4)`

## ✅ Step 0 · Enable GPU & Verify Runtime

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 50)
print(f'  PyTorch version : {torch.__version__}')
print(f'  Device          : {device}')

if torch.cuda.is_available():
    print(f'  GPU             : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('  ✅ GPU is ready!')
else:
    print('  ⚠️  No GPU detected. Go to:')
    print('     Runtime → Change runtime type → GPU (T4)')
    print('     Then restart this notebook.')
print('=' * 50)

## 📦 Step 1 · Install Dependencies

In [ ]:
# Colab already has torch, torchvision, numpy, matplotlib pre-installed.
# We only need to add the extras:
!pip install -q tqdm scikit-learn
print('✅ All dependencies ready!')

## 🗂️ Step 2 · Set Up Project Structure

In [ ]:
import os

# Create all project directories
dirs = [
    'vit_project/dataset',
    'vit_project/models',
    'vit_project/training',
    'vit_project/evaluation',
    'vit_project/utils',
    'vit_project/app',
    'vit_project/outputs/saved_models',
    'vit_project/outputs/graphs',
    'vit_project/data',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# Create __init__.py for all packages
for pkg in ['dataset', 'models', 'training', 'evaluation', 'utils', 'app']:
    open(f'vit_project/{pkg}/__init__.py', 'w').close()

print('✅ Project directory structure created:')
for d in dirs:
    print(f'   {d}/')

## ⚙️ Step 3 · Write Config File

In [ ]:
config_code = '''
import torch, os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
DATASET_NAME  = "CIFAR-10"
NUM_CLASSES   = 10
CLASS_NAMES   = ["airplane","automobile","bird","cat","deer",
                 "dog","frog","horse","ship","truck"]

# Image
IMAGE_SIZE    = 224
PATCH_SIZE    = 16
IN_CHANNELS   = 3
NUM_PATCHES   = (IMAGE_SIZE // PATCH_SIZE) ** 2   # 196

# Model (ViT-Tiny defaults — override per experiment)
EMBED_DIM     = 192
NUM_HEADS     = 3
NUM_LAYERS    = 12
MLP_DIM       = 768
DROPOUT       = 0.1

# Training
BATCH_SIZE    = 128      # Larger batch on GPU
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 1e-4
NUM_EPOCHS    = 20
WARMUP_EPOCHS = 5

# Paths
ROOT_DIR         = "/content/vit_project"
DATA_DIR         = os.path.join(ROOT_DIR, "data")
OUTPUTS_DIR      = os.path.join(ROOT_DIR, "outputs")
SAVED_MODELS_DIR = os.path.join(OUTPUTS_DIR, "saved_models")
GRAPHS_DIR       = os.path.join(OUTPUTS_DIR, "graphs")
MODEL_PATH       = os.path.join(SAVED_MODELS_DIR, "vit_model.pth")

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(GRAPHS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# CIFAR-10 normalisation stats
NORMALIZE_MEAN = (0.4914, 0.4822, 0.4465)
NORMALIZE_STD  = (0.2023, 0.1994, 0.2010)
'''

with open('vit_project/utils/config.py', 'w') as f:
    f.write(config_code)
print('✅ config.py written')

## 🧠 Step 4 · Write Vision Transformer Model

In [ ]:
model_code = '''
import torch
import torch.nn as nn
import sys, os
sys.path.insert(0, "/content/vit_project")
from utils.config import (
    IMAGE_SIZE, PATCH_SIZE, IN_CHANNELS, EMBED_DIM,
    NUM_HEADS, NUM_LAYERS, MLP_DIM, DROPOUT, NUM_CLASSES
)


class PatchEmbedding(nn.Module):
    """Split image into patches and linearly embed each patch."""
    def __init__(self, image_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                 in_channels=IN_CHANNELS, embed_dim=EMBED_DIM):
        super().__init__()
        self.patch_size  = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.projection  = nn.Conv2d(in_channels, embed_dim,
                                     kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.projection(x)   # (B, E, H/P, W/P)
        x = x.flatten(2)         # (B, E, N)
        x = x.transpose(1, 2)    # (B, N, E)
        return self.norm(x)


class MultiHeadSelfAttention(nn.Module):
    """Scaled dot-product multi-head self-attention."""
    def __init__(self, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, dropout=DROPOUT):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads
        self.scale      = self.head_dim ** -0.5
        self.qkv        = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.projection = nn.Linear(embed_dim, embed_dim)
        self.attn_drop  = nn.Dropout(dropout)
        self.proj_drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = self.attn_drop(attn.softmax(dim=-1))
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.projection(x))


class FeedForward(nn.Module):
    """Two-layer GELU MLP."""
    def __init__(self, embed_dim=EMBED_DIM, mlp_dim=MLP_DIM, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-norm Transformer encoder block."""
    def __init__(self, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 mlp_dim=MLP_DIM, dropout=DROPOUT):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff    = FeedForward(embed_dim, mlp_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    """Full Vision Transformer model."""
    def __init__(self, image_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                 in_channels=IN_CHANNELS, num_classes=NUM_CLASSES,
                 embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 num_layers=NUM_LAYERS, mlp_dim=MLP_DIM, dropout=DROPOUT):
        super().__init__()
        num_patches = (image_size // patch_size) ** 2
        self.patch_embed   = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        self.cls_token     = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_dropout   = nn.Dropout(dropout)
        self.transformer   = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, mlp_dim, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(mlp_dim // 2, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token,     std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        x = self.pos_dropout(x + self.pos_embedding)
        x = self.norm(self.transformer(x))
        return self.head(x[:, 0])

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def vit_tiny(num_classes=10):
    return VisionTransformer(embed_dim=192, num_heads=3, num_layers=12,
                             mlp_dim=768, num_classes=num_classes)

def vit_small(num_classes=10):
    return VisionTransformer(embed_dim=384, num_heads=6, num_layers=12,
                             mlp_dim=1536, num_classes=num_classes)

def vit_base(num_classes=10):
    return VisionTransformer(embed_dim=768, num_heads=12, num_layers=12,
                             mlp_dim=3072, num_classes=num_classes)
'''

with open('vit_project/models/vision_transformer.py', 'w') as f:
    f.write(model_code)
print('✅ vision_transformer.py written')

## 🗃️ Step 5 · Write Dataset Loader & Utilities

In [ ]:
dataset_code = '''
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import sys
sys.path.insert(0, "/content/vit_project")
from utils.config import IMAGE_SIZE, BATCH_SIZE, DATA_DIR, NORMALIZE_MEAN, NORMALIZE_STD

def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomCrop(IMAGE_SIZE, padding=IMAGE_SIZE // 14),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
    ])

def get_dataloaders(batch_size=BATCH_SIZE, num_workers=2, val_split=0.1):
    train_full = datasets.CIFAR10(root=DATA_DIR, train=True,
                                  download=True, transform=get_transforms(True))
    test_ds    = datasets.CIFAR10(root=DATA_DIR, train=False,
                                  download=True, transform=get_transforms(False))
    val_size   = int(len(train_full) * val_split)
    train_ds, val_ds = random_split(
        train_full, [len(train_full) - val_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    print(f"Train: {len(train_ds)} | Val: {val_size} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader
'''

with open('vit_project/dataset/cifar10_loader.py', 'w') as f:
    f.write(dataset_code)
print('✅ cifar10_loader.py written')

In [ ]:
metrics_code = '''
import torch
def accuracy(outputs, targets):
    return outputs.argmax(dim=1).eq(targets).sum().item() / targets.size(0)

def compute_epoch_metrics(running_loss, running_correct, total_samples, num_batches):
    return running_loss / num_batches, (running_correct / total_samples) * 100.0
'''
with open('vit_project/utils/metrics.py', 'w') as f:
    f.write(metrics_code)
print('✅ metrics.py written')

## 🚀 Step 6 · Load Dataset & Verify

In [ ]:
import sys
sys.path.insert(0, '/content/vit_project')

from dataset.cifar10_loader import get_dataloaders

# CIFAR-10 auto-downloads here (~170 MB)
train_loader, val_loader, test_loader = get_dataloaders(batch_size=128, num_workers=2)

images, labels = next(iter(train_loader))
print(f'Image batch : {images.shape}')   # (128, 3, 224, 224)
print(f'Label batch : {labels.shape}')   # (128,)
print('✅ CIFAR-10 loaded successfully!')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1,3,1,1)
STD  = torch.tensor([0.2023, 0.1994, 0.2010]).view(1,3,1,1)

imgs_show = (images[:16] * STD + MEAN).clamp(0, 1).permute(0,2,3,1).numpy()

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('CIFAR-10 Sample Batch  (resized to 224×224 for ViT)', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    ax.imshow(imgs_show[i])
    ax.set_title(CLASS_NAMES[labels[i].item()], fontsize=10, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/vit_project/outputs/graphs/sample_batch.png', dpi=120)
plt.show()
print('✅ Dataset visualization complete')

## 🧠 Step 7 · Initialize Vision Transformer Model

In [ ]:
import torch
from utils.config import DEVICE, IMAGE_SIZE
from models.vision_transformer import vit_tiny, vit_small

# ── Choose your model variant ────────────────────────────────
# vit_tiny  → ~5.7M params  | fastest | good for quick demo
# vit_small → ~22M params   | better accuracy | ~4x slower
# ─────────────────────────────────────────────────────────────
MODEL_VARIANT = 'tiny'   # ← change to 'small' for better accuracy

if MODEL_VARIANT == 'tiny':
    model = vit_tiny(num_classes=10)
else:
    model = vit_small(num_classes=10)

model = model.to(DEVICE)

# Sanity check: forward pass
dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(dummy)

print('═' * 55)
print(f'  Model Variant       : ViT-{MODEL_VARIANT.capitalize()}')
print(f'  Total Parameters    : {model.get_num_params():,}')
print(f'  Device              : {DEVICE}')
print(f'  Input shape         : {tuple(dummy.shape)}')
print(f'  Output shape        : {tuple(out.shape)}')
print('═' * 55)
print('✅ Model ready!')

## 🏋️ Step 8 · Train the Model

In [ ]:
import torch
import torch.nn as nn
import time
from tqdm.notebook import tqdm
from utils.config import DEVICE, MODEL_PATH, LEARNING_RATE, WEIGHT_DECAY

# ─── Hyperparameters (tune here) ─────────────────────────────
NUM_EPOCHS    = 20     # ← increase for better accuracy (30-50 recommended)
PATIENCE      = 5      # ← early stopping patience
BATCH_SIZE    = 128
LR            = 3e-4
# ──────────────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
no_improve   = 0

print(f'Training ViT-{MODEL_VARIANT.capitalize()} for {NUM_EPOCHS} epochs on {DEVICE}...')
print('─' * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── TRAIN ────────────────────────────────────────────────────
    model.train()
    train_loss = train_correct = train_total = 0

    for imgs, lbls in tqdm(train_loader, desc=f'Epoch {epoch:>2}/{NUM_EPOCHS} [Train]',
                           leave=False, ncols=80):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss    += loss.item()
        train_correct += logits.argmax(1).eq(lbls).sum().item()
        train_total   += lbls.size(0)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc  = train_correct / train_total * 100

    # ── VALIDATE ─────────────────────────────────────────────────
    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            logits      = model(imgs)
            val_loss   += criterion(logits, lbls).item()
            val_correct += logits.argmax(1).eq(lbls).sum().item()
            val_total   += lbls.size(0)

    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc  = val_correct / val_total * 100

    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_acc'].append(avg_train_acc)
    history['val_acc'].append(avg_val_acc)

    marker = ''
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        no_improve   = 0
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_acc': avg_val_acc, 'model_variant': MODEL_VARIANT}, MODEL_PATH)
        marker = '  ✅ Best saved'
    else:
        no_improve += 1

    print(f'Epoch {epoch:>2}/{NUM_EPOCHS} │ '
          f'Loss {avg_train_loss:.4f}/{avg_val_loss:.4f} │ '
          f'Acc {avg_train_acc:.1f}%/{avg_val_acc:.1f}% │ '
          f'{elapsed:.1f}s{marker}')

    if no_improve >= PATIENCE:
        print(f'\n  Early stopping at epoch {epoch} (patience={PATIENCE})')
        break

print('─' * 70)
print(f'✅ Training complete!  Best Val Acc: {best_val_acc:.2f}%')
print(f'   Model saved → {MODEL_PATH}')

## 📈 Step 9 · Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Vision Transformer – Training History', fontsize=14, fontweight='bold')

# Loss
ax1.plot(epochs, history['train_loss'], color='#3B82F6', linewidth=2, label='Train Loss')
ax1.plot(epochs, history['val_loss'],   color='#EF4444', linewidth=2, linestyle='--', label='Val Loss')
ax1.fill_between(epochs, history['train_loss'], alpha=0.1, color='#3B82F6')
ax1.fill_between(epochs, history['val_loss'],   alpha=0.1, color='#EF4444')
ax1.set_title('Loss per Epoch', fontsize=12)
ax1.set_xlabel('Epoch');  ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend();  ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(epochs, history['train_acc'], color='#10B981', linewidth=2, label='Train Acc')
ax2.plot(epochs, history['val_acc'],   color='#F59E0B', linewidth=2, linestyle='--', label='Val Acc')
ax2.fill_between(epochs, history['train_acc'], alpha=0.1, color='#10B981')
ax2.fill_between(epochs, history['val_acc'],   alpha=0.1, color='#F59E0B')
ax2.set_title('Accuracy per Epoch', fontsize=12)
ax2.set_xlabel('Epoch');  ax2.set_ylabel('Accuracy (%)')
ax2.legend();  ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/vit_project/outputs/graphs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved!')

## 🧪 Step 10 · Evaluate on Test Set

In [ ]:
import torch
import numpy as np
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from utils.config import DEVICE, MODEL_PATH, NORMALIZE_MEAN, NORMALIZE_STD
from models.vision_transformer import vit_tiny, vit_small

# ── Load best checkpoint ──────────────────────────────────────
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
print(f'Loading checkpoint from epoch {ckpt["epoch"]} (val_acc={ckpt["val_acc"]:.2f}%)')

variant = ckpt.get('model_variant', 'tiny')
eval_model = vit_tiny() if variant == 'tiny' else vit_small()
eval_model.load_state_dict(ckpt['model_state'])
eval_model = eval_model.to(DEVICE).eval()

# ── Inference ─────────────────────────────────────────────────
all_preds, all_targets = [], []
sample_imgs, sample_true, sample_pred = [], [], []

MEAN = torch.tensor(NORMALIZE_MEAN).view(1,3,1,1)
STD  = torch.tensor(NORMALIZE_STD).view(1,3,1,1)

with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc='Evaluating', ncols=80):
        preds = eval_model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_targets.extend(lbls.numpy())
        if len(sample_imgs) < 16:
            un = (imgs * STD + MEAN).clamp(0,1).permute(0,2,3,1).numpy()
            for i in range(min(len(un), 16 - len(sample_imgs))):
                sample_imgs.append(un[i])
                sample_true.append(lbls[i].item())
                sample_pred.append(preds[i].item())

# ── Metrics ────────────────────────────────────────────────────
correct  = sum(p == t for p, t in zip(all_preds, all_targets))
test_acc = correct / len(all_targets) * 100
print(f'\n Test Accuracy: {test_acc:.2f}%  ({correct}/{len(all_targets)} correct)\n')
print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, digits=4))

## 🗃️ Step 11 · Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_targets, all_preds)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(10));  ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(CLASS_NAMES, fontsize=11)

# Annotate cells
thresh = cm.max() / 2.
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > thresh else 'black')

ax.set_title('Confusion Matrix – CIFAR-10', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('/content/vit_project/outputs/graphs/confusion_matrix.png', dpi=150)
plt.show()
print('✅ Confusion matrix saved')

## 🖼️ Step 12 · Sample Predictions Grid

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Sample Test Predictions  (Green = Correct · Red = Wrong)',
             fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    img = np.clip(sample_imgs[i], 0, 1)
    correct = sample_true[i] == sample_pred[i]
    color   = 'green' if correct else 'red'

    ax.imshow(img)
    ax.set_title(f'True: {CLASS_NAMES[sample_true[i]]}\nPred: {CLASS_NAMES[sample_pred[i]]}',
                 color=color, fontsize=9, fontweight='bold')
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)
        spine.set_visible(True)

correct_patch = mpatches.Patch(color='green', label='Correct')
wrong_patch   = mpatches.Patch(color='red',   label='Incorrect')
fig.legend(handles=[correct_patch, wrong_patch], loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('/content/vit_project/outputs/graphs/sample_predictions.png', dpi=150)
plt.show()
print('✅ Sample predictions saved')

## 🎯 Step 13 · Predict a Custom Image

In [ ]:
# ─── Option A: Upload your own image ────────────────────────────────────────
from google.colab import files as colab_files
print('Upload an image to classify (jpg / png / webp):')
uploaded = colab_files.upload()
IMAGE_PATH = list(uploaded.keys())[0]
print(f'Uploaded: {IMAGE_PATH}')

In [ ]:
# ─── Option B: Use a URL instead (comment out Option A above) ───────────────
# import requests
# from io import BytesIO
# URL        = 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg'
# IMAGE_PATH = '/content/test_image.jpg'
# with open(IMAGE_PATH, 'wb') as f:
#     f.write(requests.get(URL).content)

# ─── Run prediction ─────────────────────────────────────────────────────────
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

INFERENCE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2023, 0.1994, 0.2010)),
])

img_pil  = Image.open(IMAGE_PATH).convert('RGB')
tensor   = INFERENCE_TRANSFORM(img_pil).unsqueeze(0).to(DEVICE)

eval_model.eval()
with torch.no_grad():
    logits = eval_model(tensor)
    probs  = torch.softmax(logits, dim=1)[0]

topk_probs, topk_idx = probs.topk(5)
results = [
    {'class': CLASS_NAMES[idx.item()], 'confidence': prob.item() * 100}
    for idx, prob in zip(topk_idx, topk_probs)
]

# ─── Visualise ──────────────────────────────────────────────────────────────
CLASS_ICONS = {'airplane':'✈️','automobile':'🚗','bird':'🐦','cat':'🐱',
               'deer':'🦌','dog':'🐶','frog':'🐸','horse':'🐴','ship':'🚢','truck':'🚚'}

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(14, 5),
                                      gridspec_kw={'width_ratios': [1, 1.5]})
fig.suptitle('Vision Transformer  –  Custom Image Prediction', fontsize=13, fontweight='bold')

ax_img.imshow(img_pil.resize((224, 224)))
ax_img.set_title(
    f"{CLASS_ICONS.get(results[0]['class'],'🎯')}  Predicted: {results[0]['class'].upper()}\n"
    f"Confidence: {results[0]['confidence']:.1f}%",
    fontsize=12, fontweight='bold', color='#1e40af'
)
ax_img.axis('off')

labels_r = [f"{CLASS_ICONS.get(r['class'],'🎯')} {r['class'].capitalize()}" for r in results]
confs_r  = [r['confidence'] for r in results]
colors_r = ['#6366f1' if i == 0 else '#94a3b8' for i in range(len(results))]

bars = ax_bar.barh(labels_r[::-1], confs_r[::-1], color=colors_r[::-1], edgecolor='none', height=0.55)
for bar, conf in zip(bars, confs_r[::-1]):
    ax_bar.text(conf + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{conf:.1f}%', va='center', ha='left', fontsize=10)
ax_bar.set_xlim(0, 110)
ax_bar.set_xlabel('Confidence (%)', fontsize=11)
ax_bar.set_title('Top-5 Class Probabilities', fontsize=11)
ax_bar.grid(axis='x', alpha=0.3)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/content/vit_project/outputs/graphs/custom_prediction.png', dpi=150)
plt.show()

print('\n  Top-5 Predictions:')
print('  ' + '─' * 40)
for r in results:
    bar = '█' * int(r['confidence'] / 5)
    print(f"  {r['class']:>12} : {r['confidence']:6.2f}%  {bar}")
print(f"\n  ▶  Final Answer → {results[0]['class'].upper()} ({results[0]['confidence']:.1f}%)")

## 💾 Step 14 · Save Everything to Google Drive

In [ ]:
from google.colab import drive
import shutil, os

# Mount Google Drive
drive.mount('/content/drive')

# Create destination folder
DRIVE_DIR = '/content/drive/MyDrive/ViT_CIFAR10_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy model checkpoint
shutil.copy('/content/vit_project/outputs/saved_models/vit_model.pth',
            f'{DRIVE_DIR}/vit_model.pth')

# Copy all plots
graphs_src = '/content/vit_project/outputs/graphs/'
for fname in os.listdir(graphs_src):
    if fname.endswith('.png'):
        shutil.copy(os.path.join(graphs_src, fname),
                    os.path.join(DRIVE_DIR, fname))

print(f'✅ All results saved to Google Drive: {DRIVE_DIR}')
print('\nSaved files:')
for f in os.listdir(DRIVE_DIR):
    size = os.path.getsize(f'{DRIVE_DIR}/{f}') / 1024
    print(f'   {f:<40} {size:>8.1f} KB')

In [ ]:
# ─── OR: Download directly to your PC (no Drive needed) ───────────────────
from google.colab import files
import os

# Download model
files.download('/content/vit_project/outputs/saved_models/vit_model.pth')

# Download all graph PNGs
graphs_dir = '/content/vit_project/outputs/graphs/'
for fname in os.listdir(graphs_dir):
    if fname.endswith('.png'):
        files.download(os.path.join(graphs_dir, fname))

print('✅ Files downloaded to your PC!')

## 📊 Step 15 · Final Summary
---

In [ ]:
print('╔══════════════════════════════════════════════════════╗')
print('║       Vision Transformer – Colab Run Summary         ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Model Variant      : ViT-{MODEL_VARIANT.capitalize():<26}║')
print(f'║  Parameters         : {model.get_num_params():>10,}                 ║')
print(f'║  Device Used        : {str(DEVICE):<29}║')
print(f'║  Epochs Trained     : {len(history["train_loss"]):<29}║')
print(f'║  Best Val Accuracy  : {best_val_acc:>6.2f}%                       ║')
print(f'║  Test Accuracy      : {test_acc:>6.2f}%                       ║')
print('╠══════════════════════════════════════════════════════╣')
print('║  Files Saved:                                        ║')
print('║   • vit_model.pth          (trained weights)         ║')
print('║   • training_curves.png    (loss + accuracy)         ║')
print('║   • confusion_matrix.png   (10×10 heatmap)           ║')
print('║   • sample_predictions.png (prediction grid)         ║')
print('╚══════════════════════════════════════════════════════╝')